In [ ]:
import random
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms

# == Import your models ==
from models.pv_model import run_pvsam
from models.csp_model import run_basic_mspt
from models.dispatch_opt import optimize_dispatch, calculate_firm_bonus
from capex_opex.excel_capex_opex import calc_capex_opex

# ------------------------------------------------------------------------------------
# Config
# ------------------------------------------------------------------------------------
excel_file = Path("capex_opex/CAPEX OPEX model CSP-PV MJ2500.xlsx")
random.seed(42)

# Bounds for decision variables: [desired_array_size, P_ref, t_TES_hours]
BOUNDS = [
    (50_000, 250_000),   # PV size (kWdc)
    (25.0, 150.0),       # CSP size (MWe)
    (4, 20)              # TES hours
]

TRIAL_LOG = []

# Toggle to use textbook-style discounting windows
USE_TEXTBOOK_FINANCIALS = True

# Financial assumptions
debt = 0.8
equity = 0.2
debt_rate = 0.05
equity_rate = 0.08
tax_rate = 0.25
lifetime_years = 30
construction_years = 2

# ------------------------------------------------------------------------------------
# System evaluation (cached)
# ------------------------------------------------------------------------------------
@lru_cache(maxsize=256)
def eval_system(desired_array_size, P_ref, t_TES_hours):
    """
    Returns:
        PPA (float), -CF_hybrid (float), g0 (float: feasibility margin)
    """
    # Fixed tech parameters
    dc_to_ac_ratio = 1.2
    solarm = 3
    eta_PB = 0.40
    max_to_grid = 100

    # Cast types (important for cache consistency)
    desired_array_size = int(desired_array_size)
    P_ref = float(P_ref)
    t_TES_hours = int(t_TES_hours)

    # --- PV model ---
    df_pv, pv_summary, pv_obj, annual_energy_kwh, land_area_m2, name_plate_kwdc = run_pvsam(
        system_kwargs=dict(desired_array_size=desired_array_size, dc_to_ac_ratio=dc_to_ac_ratio)
    )

    # --- CSP model ---
    df_csp, csp_summary, mspt = run_basic_mspt(P_ref=P_ref, solarm=solarm)
    q_csp = df_csp['P_rec_MWt_calculated'].tolist()           # MW_th receiver power vs time
    e_pv = (df_pv['AC_kWh'] / 1000).tolist()                  # MW_e from PV vs time

    # --- Dispatch optimization (hybrid) ---
    res, model, E_cap, PV_to_heater_max, CF_hybrid, CF_pb = optimize_dispatch(
        q_csp, e_pv,
        eta_PB=eta_PB,
        x_price=1.0,
        max_to_grid=max_to_grid,
        allow_spill=True,
        objective_mode="revenue",
        PB_e_max=csp_summary.get("NetCapacity_MWe"),
        E_cap=(csp_summary.get("NetCapacity_MWe") * t_TES_hours) / eta_PB,  # MWh_th capacity
        solarm=solarm,
    )

    # --- Firm energy bonus model ---
    bonus = calculate_firm_bonus(res, base_price=50.0, bonus_rate=0.2, margin=0.05, min_hours=3)

    # --- CAPEX / OPEX inputs for Excel model ---
    inputs = {
        "PB Installed Capacity (Gross)": csp_summary.get("NetCapacity_MWe") * 1000,      # kWe
        "Solar Field Aperture Area (Mirror Area)": csp_summary.get("Solar_Field_Area_m2"),
        "Thermal Energy Storage Capacity ": E_cap,                                       # MWh_th
        "Receiver Power (Max Rated)": csp_summary.get("Receiver_Design_MWt"),
        "Tower Height (w/o Receiver)": csp_summary.get("Tower_Height_m"),
        "Electric Heater Thermal Power (Max Rated)": PV_to_heater_max * 1000,            # kW_th
        "Land Area CSP": csp_summary.get("Land_Area_acre") * 4046.86,                    # m2
        "PV Installed Capacity": name_plate_kwdc * 1000,                                 # kWdc
        "Battery Pack Power (Max Rated)": 0,
        "Battery Pack Capacity": 0,
        "Battery Annual Generation (for OPEX)": 0,
        "Land Area PV": land_area_m2,
        "CSP Annual Generation (for OPEX)": csp_summary.get("Annual_kWh") / 1e6,         # MWh
        "Distance to Grid ": 0,
        "Distance to Road": 0,
        "Distance to Gas": 0,
        "Distance to Water": 0,
        "Other Component Size": 0,
    }
    overrides = {
        "BOP": 74.8,
        "SF_aperture": 115,
        "PV_modules": 0.3,
        "pv_fixed_opex_coeff": 1,
    }

    capex_musd, opex_musd, capex_df, opex_df = calc_capex_opex(
        excel_file, inputs, return_breakdowns=True, ref_cost_overrides=overrides
    )

    # --- Finance: WACC, sigma factors, and PPA ---
    wacc = (debt * debt_rate * (1 - tax_rate)) + (equity * equity_rate)

    if USE_TEXTBOOK_FINANCIALS:
        # Capex spread equally over construction years and discounted to t=0
        sigma_capex = sum((capex_musd * 1e6 / construction_years) / (1 + wacc) ** t
                          for t in range(1, construction_years + 1))
        # Opex discounted over operating years only (post-COD)
        sigma_opex = sum(1 / (1 + wacc) ** t for t in range(1, lifetime_years + 1))
    else:
        # Your original approach (kept for reproducibility)
        sigma_capex = sum(
            (capex_musd * 1e6) / (construction_years * (1 + wacc) ** t)
            for t in range(0, construction_years - 1)
        )
        sigma_opex = sum(1 / (1 + wacc) ** t for t in range(1, lifetime_years + construction_years - 1))

    A = sigma_capex / sigma_opex  # CRF-like annuity
    # Denominator should represent yearly "energy or revenue units" consistent with bonus model
    # Here we keep your denominator (bonus-adjusted revenue proxy)
    PPA = (A + opex_musd * 1e6) / bonus['total_revenue_with_bonus']

    result = {
        "desired_array_size": desired_array_size,
        "P_ref": P_ref,
        "t_TES_hours": t_TES_hours,
        "PPA": float(PPA),
        "CF_hybrid": float(CF_hybrid),
        "CF_pb": float(CF_pb),
    }
    TRIAL_LOG.append(result)

    # Constraint: P_ref ≥ 0.25 * (desired_array_size / 1000)
    csp_min = 0.25 * (desired_array_size / 1000.0)
    g0 = P_ref - csp_min

    # Objectives returned to DEAP driver
    return float(PPA), -float(CF_hybrid), float(g0)

# ------------------------------------------------------------------------------------
# DEAP setup
# ------------------------------------------------------------------------------------
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0))  # min PPA, min -CF (max CF)
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_pv", random.randint, BOUNDS[0][0], BOUNDS[0][1])
toolbox.register("attr_csp", random.uniform, BOUNDS[1][0], BOUNDS[1][1])
toolbox.register("attr_tes", random.randint, BOUNDS[2][0], BOUNDS[2][1])
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_pv, toolbox.attr_csp, toolbox.attr_tes), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(ind):
    ppa, ncf, g0 = eval_system(int(ind[0]), float(ind[1]), int(ind[2]))
    # DeltaPenalty requires returning only objectives
    return (ppa, ncf)

def feasible(ind):
    # Algebraic check without re-running models
    desired_array_size = float(ind[0])
    P_ref = float(ind[1])
    csp_min = 0.25 * (desired_array_size / 1000.0)
    return (P_ref - csp_min) >= 0.0

toolbox.register("evaluate", evaluate)
toolbox.decorate("evaluate", tools.DeltaPenalty(feasible, 1e12))

toolbox.register("mate", tools.cxBlend, alpha=0.3)
toolbox.register("mutate", tools.mutPolynomialBounded,
                 low=[b[0] for b in BOUNDS],
                 up=[b[1] for b in BOUNDS],
                 eta=20.0, indpb=1.0/3)
toolbox.register("select", tools.selNSGA2)

# Optional: ensure integer genes stay integers and clamp to bounds after operators
def _repair(ind):
    # Integer genes
    ind[0] = int(round(ind[0]))  # PV size (kWdc)
    ind[2] = int(round(ind[2]))  # TES hours
    # Clamp all genes to bounds
    for i, (lo, hi) in enumerate(BOUNDS):
        if ind[i] < lo: ind[i] = lo
        if ind[i] > hi: ind[i] = hi
    return ind

# ------------------------------------------------------------------------------------
# Run GA
# ------------------------------------------------------------------------------------
def run_ga(pop_size=32, ngen=12, cxpb=0.9, mutpb=0.2, save_trials="deap_nsga_trials.csv",
           save_pareto="deap_pareto_front.csv"):
    pop = toolbox.population(n=pop_size)

    # Initialize population fitness (required by selNSGA2 to compute crowding dist)
    invalid_ind = [ind for ind in pop if not ind.fitness.valid]
    fits = list(map(toolbox.evaluate, invalid_ind))
    for ind, fit in zip(invalid_ind, fits):
        ind.fitness.values = fit
    pop = toolbox.select(pop, k=len(pop))  # assign crowding distances

    for gen in range(ngen):
        offspring = algorithms.varAnd(pop, toolbox, cxpb, mutpb)
        offspring = [_repair(ind) for ind in offspring]

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fits = list(map(toolbox.evaluate, invalid_ind))
        for ind, fit in zip(invalid_ind, fits):
            ind.fitness.values = fit

        pop = toolbox.select(pop + offspring, k=len(pop))

    # Save all parameter trials/output
    df = pd.DataFrame(TRIAL_LOG)
    df.to_csv(save_trials, index=False)

    # Pareto front output
    pareto = tools.sortNondominated(pop, k=len(pop), first_front_only=True)[0]
    pareto_data = [[int(round(ind[0])), float(ind[1]), int(round(ind[2])),
                    float(ind.fitness.values[0]), -float(ind.fitness.values[1])]
                   for ind in pareto]
    df_pareto = pd.DataFrame(pareto_data, columns=['pv_size_kWdc', 'csp_size_MWe', 'tes_h', 'ppa', 'CF_hybrid'])
    df_pareto.to_csv(save_pareto, index=False)

    return pop, df, df_pareto

# ------------------------------------------------------------------------------------
# Main
# ------------------------------------------------------------------------------------
if __name__ == "__main__":
    pop, df_trials, df_pareto = run_ga()
    print("Done. Saved:")
    print(" - Trials:", "deap_nsga_trials.csv")
    print(" - Pareto:", "deap_pareto_front.csv")


CORETAN

In [ ]:
from models.pv_model import run_pvsam
import numpy as np
from capex_opex.excel_capex_opex import calc_capex_opex
from pathlib import Path
from models.csp_model import run_basic_mspt
import sys
sys.path.append(".")
from models.dispatch_opt import optimize_dispatch, export_dispatch_to_csv,calculate_firm_bonus

excel_file = Path("capex_opex/CAPEX OPEX model CSP-PV MJ2500.xlsx")
# == PV ===
desired_array_size = 102000     # kWdc
dc_to_ac_ratio=1.2,             # DC to AC ratio for PV system
# == CSP ===
P_ref = 100                   # CSP plant size in MWe
solarm = 3                      # Solar Multiple
eta_PB=0.40                     # PB efficiency
# == TES ===
t_TES_hours = 16                # hours of thermal energy storage capacity
max_to_grid = 100               # MWe limit
# == ECONOMIC MODEL ===
x_price = 1                     
debt = 0.8
equity = 0.2
debt_rate = 0.05
equity_rate = 0.08
tax_rate = 0.25
lifetime_years = 30
construction_years = 2
# == MISC ===
dist_to_grid = 0               # km
dist_to_road = 0                # km
dist_to_gas = 0                # km
dist_to_water = 0              # km
other_comp = 0                  # dummy variable for other component size

df_pv, pv_summary, pv_obj, annual_energy_kwh, land_area_m2, name_plate_kwdc = run_pvsam(
    system_kwargs=dict(
        desired_array_size=desired_array_size,  
        dc_to_ac_ratio=dc_to_ac_ratio,
    )
)

df_csp, csp_summary, mspt = run_basic_mspt(
    P_ref=P_ref,
    solarm=solarm,
)

q_csp = df_csp['P_rec_MWt_calculated'].tolist()  
e_pv  = (df_pv['AC_kWh'] / 1000).tolist() 

res, model, E_cap, PV_to_heater_max,CF_hybrid,CF_pb = optimize_dispatch(
    q_csp, 
    e_pv, 
    eta_PB=eta_PB,
    x_price=x_price,
    max_to_grid=max_to_grid,   
    allow_spill=True,     
    objective_mode="revenue",         # <---: "revenue" or "cf_hybrid" or "cf_pb" (optional)                       
    PB_e_max=csp_summary.get("NetCapacity_MWe"),
    E_cap=(csp_summary.get("NetCapacity_MWe")*t_TES_hours)/eta_PB,
    solarm=solarm,
    )  

bonus = calculate_firm_bonus(res, base_price=50.0, bonus_rate=0.2, margin=0.05, min_hours=3)

inputs = {
    "PB Installed Capacity (Gross)":            csp_summary.get("NetCapacity_MWe")*1000,                # kW
    "Solar Field Aperture Area (Mirror Area)":  csp_summary.get("Solar_Field_Area_m2"),                 # m2
    "Thermal Energy Storage Capacity ":         E_cap,
    "Receiver Power (Max Rated)":               csp_summary.get("Receiver_Design_MWt"),                 # MW 0 if Parabolic Trough
    "Tower Height (w/o Receiver)":              csp_summary.get("Tower_Height_m"),                      # m
    # "Electric Heater Thermal Power (Max Rated)":csp_summary.get("Electric_Heater_Power_MWt"),         # MWt
    "Electric Heater Thermal Power (Max Rated)":PV_to_heater_max*1000,
    "Land Area CSP":                            csp_summary.get("Land_Area_acre") * 4046.86,            # m2
    "PV Installed Capacity":                    name_plate_kwdc *1000,                                  # Wdc 
    "Battery Pack Power (Max Rated)":           0,                                                    # MW bess_energy_mwh or 
    "Battery Pack Capacity":                    0,                                                      # MWh-e bess_energy_mwh or 
    "Battery Annual Generation (for OPEX)":     0,                                                    # GWh/y
    "Land Area PV":                             land_area_m2,                                           # m2
    "CSP Annual Generation (for OPEX)":         csp_summary.get("Annual_kWh") / 1000000,                # GWh/y
    "Distance to Grid ":                        dist_to_grid,
    "Distance to Road":                         dist_to_road,
    "Distance to Gas":                          dist_to_gas,
    "Distance to Water":                        dist_to_water,
    "Other Component Size":                     other_comp,
}

overrides = {
    # If tower technology: SET BOP Reference to 74.8 MUSD or If PT technology: SET BOP Reference Cost to 90.2 MUSD
    "BOP": 74.8,          # MUSD (replaces CAPEX!D27)
    # If tower technology: SET Solar Field Reference Cost to 125 MUSD or If PT technology: SET Solar Field Reference Cost to 115 MUSD
    "SF_aperture": 115,  # MUSD (replaces CAPEX!D28)
    # pv_modules_ref_cost_musd=0.33,# $/Wdc
    "PV_modules": 0.3,   # MUSD (replaces CAPEX!D41)
    #Fixed Trackers	0.01 ; Single Axis Tracking	0.0175
    "pv_fixed_opex_coeff": 1,
}

capex_musd, opex_musd, capex_df, opex_df = calc_capex_opex(
    excel_file, inputs, return_breakdowns=True, ref_cost_overrides=overrides
)

wacc = (debt * debt_rate * (1 - tax_rate)) + (equity * equity_rate)
sigma_capex = sum((capex_musd*10**6) / (construction_years*(1 + wacc)**t) for t in range(0, construction_years - 1))
sigma_opex = sum(1 / (1 + wacc)**t for t in range(1,lifetime_years + construction_years -1))
A = sigma_capex / sigma_opex
PPA = (A+opex_musd*10**6) / bonus['total_revenue_with_bonus']

revenue = res.get("revenue_total", None)
if revenue is None:
    # compute from outputs
    price = np.array(res["price_profile"], dtype=float)           # EUR/MWh
    export = (np.array(res["pv_to_grid_MWe"], dtype=float) +
              np.array(res["pb_electric_MWe"], dtype=float))      # MWe
    dt_h = 1.0  # adjust if you solved with other dt
    revenue = float((price * export * dt_h).sum())


